In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import random

### Setting up environment
- making the frozen lake environment as non slippery to improve the prbability
  - since with slippery it would be 1/3
  -  this makes the environment determinitstic

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")

In [ ]:
action_size = env.action_space.n
state_size = env.observation_space.n

#### Q table

In [ ]:
qtable = np.zeros((state_size, action_size))
print(qtable)

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


### Setting up Hyper parameters

In [ ]:
total_episodes = 1500         # Total episodes
max_steps = 20                # Max steps per episode
learning_rate = 0.8           # Learning rate
gamma = 0.95                  # Discounting rate

### Exploration

In [ ]:
epsilon = 1.0                 # Exploration rate
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.1            # Minimum exploration probability
decay_rate = 0.01

### Training with Q-Learning

In [ ]:
rewards = []
for episode in range(total_episodes):
    state, info = env.reset()
    step = 0
    done = False
    total_rewards = 0

    for step in range(max_steps):
        exp_exp_tradeoff = random.uniform(0, 1)
        if exp_exp_tradeoff > epsilon:
            action = np.argmax(qtable[state, :])
        else:
            action = env.action_space.sample()

        new_state, reward, done, truncated, info = env.step(action)
        qtable[state, action] = qtable[state, action] + learning_rate * (
            reward + gamma * np.max(qtable[new_state, :]) - qtable[state, action]
        )
        total_rewards += reward
        state = new_state
        if done or truncated:
            break

    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
    rewards.append(total_rewards)

print("Training completed.")
print("Score over time (success rate):", sum(rewards) / total_episodes)
print("Q-table:")
print(qtable)

# Evaluation loop with debugging
eval_max_steps = 50  # Increased for evaluation to ensure termination
for episode in range(5):
    state, info = env.reset()
    step = 0
    done = False
    print("****************************************************")
    print("EPISODE", episode)

    for step in range(eval_max_steps):
        action = np.argmax(qtable[state, :])
        new_state, reward, done, truncated, info = env.step(action)
        if done or truncated:
            env.render()
            print("Number of steps:", step)
            print("Reward:", reward)
            break
        state = new_state
    else:  # If loop completes without breaking (i.e., no termination)
        print("Episode did not terminate within", eval_max_steps, "steps.")
        print("Final state:", state)

env.close()